# KrishiNexus · potato grade check for `yolov11n-krishi-bd-v2.1` (run 2)

Run 2 of the potato vision notebook trained and chose the defect segmenter, but its last step crashed: CUDA ran out
of memory while predicting at 1024 px with full-resolution masks. "Grade accuracy" was never measured. This notebook
runs that pre-registered check once, on CPU, with nothing retuned.

- **Model:** run 2's chosen `defect_best.pt` (YOLO11n-seg at 1024 px), read from that notebook's output and checked
  against its SHA-256.
- **Check:** on each of the 267 test crops, the predicted defect masks become "bruising %" (scab, greening and
  mechanical damage) and "rot %". The app's BARI rule turns them into grade A, B or C. That grade is compared with the
  grade from the hand-drawn labels, and with a majority-grade baseline from the train split. Target (set before
  run 1): 0.70.
- **Same mask path as the app:** predicted masks are rasterised from their `Results.masks.xy` outlines at the crop's
  own size, as `optical_grader.fill_outlines` does. So the number describes what the app runs, and no full-size mask
  stack is ever built.
- **Samples:** six original validation photos (a seeded pick, bytes untouched, EXIF kept) and their labels, so the
  app's two-stage cascade can be tested on real market photos.

**Data.** Tasnia, M. H. (2026). *Diamant Potato Dataset for ROI and Surface Defect Instance Segmentation*. Mendeley
Data, V1. doi:10.17632/4vty9rz7zw.1. Daffodil International University, CC BY 4.0.

**Before you run:** Settings → Internet → on (the data and `pip`). No accelerator is needed. Attach the output of
`krishinexus-potato-vision-training` (run 2) as an input.

**Frozen-test rule:** run 2 already opened the defect test split for its mask mAP, and this check belongs to that same
pre-registered evaluation. The model, image size and threshold are run 2's. Nothing is tuned against the result.


## 1. Config


In [ ]:
import hashlib, json, os, pathlib, random, shutil, subprocess, sys, time, zipfile

SEED = 20261109
WORK = "/kaggle/working"
RAW = "/tmp/diamant_raw"            # downloads and extracted zips (not kept)
DATA = "/tmp/diamant_defect"        # the defect set, stored as run 2 stored it (not kept)
MAX_SIDE = 1024                     # run 2 resized every image to this before training and scoring
DEFECT_IMGSZ = 1024                 # run 2's chosen candidate, n1024
CONF = 0.25                         # run 2's detection threshold for the grading check
TARGET = 0.70                       # grade accuracy, set before run 1 (TRAINING_PLAN.md section 8)
N_SAMPLES = 6
DEFECT_SHA256 = "84a079d7bbf331f42a6113c90e098fa9abe156b9693b4224acabf80aada6198d"   # run 2's defect_best.pt

ZIPS = {   # Mendeley Data file links and their published SHA-256
    "defect": ("https://data.mendeley.com/public-files/datasets/4vty9rz7zw/files/4863d50a-390b-474b-9ef8-482a42b7bc33/file_downloaded",
               "89cbdfc1072415e35ec11fc4a22318a848fa90285b1a927112438af1c1bc3d48"),
    "roi": ("https://data.mendeley.com/public-files/datasets/4vty9rz7zw/files/187b5ef3-d642-4cbd-9cef-3dcab2d84ddd/file_downloaded",
            "27a32167f8c71b88c8ec88f80249b91a3a019ecc1992ed6e95f67fac262cea0d"),
}


## 2. Helpers (the vision notebook's, self-tested in the repo)


In [ ]:
"""Helpers for the Diamant potato vision notebook: dataset layout, YOLO labels and the grading metrics."""
import pathlib

import numpy as np

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
SPLIT_NAMES = {"train": "train", "training": "train", "val": "val", "valid": "val", "validation": "val",
               "test": "test", "testing": "test"}
DEFECT_NAMES = ["scab", "rot", "greening", "mechanical_damage"]   # class ids 0-3, as published on the dataset page
BLEMISH, ROT = (0, 2, 3), (1,)   # the grader's two inputs: surface blemish feeds "bruising %", decay feeds "rot %"
GRADE_LIMITS = {"a_bruising": 3, "a_rot": 2, "b_bruising": 8, "b_rot": 5}   # prompt_registry "bari-standard"


def split_of(path):
    """The split named by the nearest directory in the path, or None."""
    for part in reversed(pathlib.PurePath(path).parts[:-1]):
        if part.lower() in SPLIT_NAMES:
            return SPLIT_NAMES[part.lower()]
    return None


def label_path(image):
    """YOLO convention: swap the nearest 'images' directory for 'labels' and use a .txt suffix."""
    parts = list(pathlib.PurePath(image).parts)
    for i in range(len(parts) - 2, -1, -1):
        if parts[i].lower() == "images":
            parts[i] = "labels"
            break
    return pathlib.Path(*parts).with_suffix(".txt")


def pair(files):
    """{split: [(image, label path or None)]} for every image, skipping macOS resource forks."""
    labels = {str(f) for f in files if f.suffix.lower() == ".txt"}
    out = {}
    for f in files:
        if f.suffix.lower() in IMAGE_EXTS and "__MACOSX" not in f.parts and not f.name.startswith("._"):
            lab = label_path(f)
            out.setdefault(split_of(f), []).append((f, lab if str(lab) in labels else None))
    return out


def read_label(text):
    """[(class_id, coords)] from a YOLO label: 4 numbers are a box, an even number above 4 a polygon."""
    rows = []
    for line in text.splitlines():
        tok = line.split()
        if not tok:
            continue
        nums = [float(t) for t in tok[1:]]
        if len(nums) < 4 or (len(nums) > 4 and len(nums) % 2):
            raise ValueError(f"bad YOLO label line: {line[:80]}")
        rows.append((int(float(tok[0])), np.array(nums, dtype=np.float32)))
    return rows


def polygon_mask(rows, classes, h, w):
    """Union mask of the labelled shapes whose class is in `classes`; boxes are drawn as rectangles."""
    from PIL import Image, ImageDraw

    img = Image.new("1", (w, h), 0)
    draw = ImageDraw.Draw(img)
    for cls, c in rows:
        if cls not in classes:
            continue
        if len(c) == 4:
            x, y, bw, bh = c
            c = np.array([x - bw / 2, y - bh / 2, x + bw / 2, y - bh / 2, x + bw / 2, y + bh / 2, x - bw / 2, y + bh / 2])
        pts = [(float(px) * w, float(py) * h) for px, py in c.reshape(-1, 2)]
        if len(pts) >= 3:
            draw.polygon(pts, fill=1)
    return np.array(img, dtype=bool)


def union(masks, cls, classes, shape):
    """Union of the predicted masks whose class is in `classes` (an empty mask when there are none)."""
    keep = np.isin(cls, classes)
    return masks[keep].any(axis=0) if keep.any() else np.zeros(shape, dtype=bool)


def area_pct(mask, potato):
    """Share of the potato's visible pixels that the mask covers, in percent."""
    n = int(potato.sum())
    return 100.0 * float((mask & potato).sum()) / n if n else 0.0


def grade(bruising, rot, lim=GRADE_LIMITS):
    """The app's grade rule for one potato (uniformity is a lot-level measure, so it is left out)."""
    if bruising <= lim["a_bruising"] and rot <= lim["a_rot"]:
        return "A"
    if bruising <= lim["b_bruising"] and rot <= lim["b_rot"]:
        return "B"
    return "C"


def presence(gt_sets, pred_sets, n_classes):
    """Per class, precision/recall/F1 of the image-level call "this potato shows class c"."""
    out = {}
    for c in range(n_classes):
        tp = sum(c in g and c in p for g, p in zip(gt_sets, pred_sets))
        fp = sum(c not in g and c in p for g, p in zip(gt_sets, pred_sets))
        fn = sum(c in g and c not in p for g, p in zip(gt_sets, pred_sets))
        prec = tp / (tp + fp) if tp + fp else None
        rec = tp / (tp + fn) if tp + fn else None
        f1 = None if prec is None or rec is None else (2 * prec * rec / (prec + rec) if prec + rec else 0.0)
        out[c] = {"precision": prec, "recall": rec, "f1": f1, "support": tp + fn}
    return out


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)
import cv2, torch, ultralytics
from ultralytics import YOLO
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| opencv", cv2.__version__)


## 3. Data: download, verify, store the defect set as run 2 did, keep six validation photos


In [ ]:
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from PIL import Image, ImageOps

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 23), b""):
            h.update(block)
    return h.hexdigest()

pathlib.Path(RAW).mkdir(parents=True, exist_ok=True)
t0 = time.time()
for name, (url, digest) in ZIPS.items():
    zpath = pathlib.Path(RAW, f"{name}.zip")
    subprocess.run(["curl", "-L", "--fail", "--silent", "--show-error", "--retry", "5", "--retry-delay", "15",
                    "-o", str(zpath), url], check=True)
    got = sha256(zpath)
    assert got == digest, f"{name}.zip has sha256 {got}, but the dataset page publishes {digest}"
    with zipfile.ZipFile(zpath) as z:
        z.extractall(pathlib.Path(RAW, name))
    zpath.unlink()
    print(f"{name}: downloaded, sha256 verified, extracted ({time.time() - t0:.0f} s)")

def convert(job):   # run 2's conversion, unchanged: EXIF-upright, at most MAX_SIDE, crops kept lossless
    split, img, lab = job
    im = Image.open(img)
    if im.format == "JPEG":
        im.draft("RGB", (MAX_SIDE, MAX_SIDE))
    im = ImageOps.exif_transpose(im).convert("RGB")
    im.thumbnail((MAX_SIDE, MAX_SIDE), Image.BILINEAR)
    jpg = img.suffix.lower() in (".jpg", ".jpeg")
    dst = pathlib.Path(DATA, "images", split, img.stem + (".jpg" if jpg else ".png"))
    im.save(dst, quality=92) if jpg else im.save(dst, compress_level=1)
    pathlib.Path(DATA, "labels", split, img.stem + ".txt").write_text(lab.read_text() if lab else "")

files = lambda name: sorted(p for p in pathlib.Path(RAW, name).rglob("*") if p.is_file())
defect = pair(files("defect"))
assert {s: len(defect.get(s, [])) for s in ("train", "test")} == {"train": 1281, "test": 267}, "defect split sizes differ from run 2"
for split in ("train", "test"):
    for sub in ("images", "labels"):
        pathlib.Path(DATA, sub, split).mkdir(parents=True, exist_ok=True)
with ThreadPoolExecutor(8) as ex:
    list(ex.map(convert, [(s, img, lab) for s in ("train", "test") for img, lab in defect[s]]))

roi_val = sorted(pair(files("roi"))["val"], key=lambda p: p[0].name)
assert len(roi_val) == 477, len(roi_val)
SAMPLES = pathlib.Path(WORK, "samples")
SAMPLES.mkdir(exist_ok=True)
for img, lab in random.Random(SEED).sample(roi_val, N_SAMPLES):
    shutil.copy(img, SAMPLES / img.name)
    if lab:
        shutil.copy(lab, SAMPLES / (img.stem + ".txt"))
print("samples:", sorted(p.name for p in SAMPLES.iterdir()))


## 4. The grading check (test split, once)


In [ ]:
def fill_outlines(shape, outlines):   # optical_grader.fill_outlines: rasterise Results.masks.xy into one mask
    mask = np.zeros(shape, np.uint8)
    polygons = [np.round(o).astype(np.int32) for o in outlines if len(o) >= 3]
    if polygons:
        cv2.fillPoly(mask, polygons, 1)
    return mask.astype(bool)

def gt_areas(img_path, split):   # run 2's, unchanged
    potato = np.array(Image.open(img_path).convert("RGB")).max(axis=2) > 10   # crops sit on uniform black
    rows = read_label(pathlib.Path(DATA, "labels", split, pathlib.Path(img_path).stem + ".txt").read_text())
    h, w = potato.shape
    return rows, potato, [area_pct(polygon_mask(rows, g, h, w), potato) for g in (BLEMISH, ROT)]

weights = next(pathlib.Path("/kaggle/input").rglob("defect_best.pt"))
assert sha256(weights) == DEFECT_SHA256, f"{weights} is not run 2's defect_best.pt"
model = YOLO(str(weights))
train_grades = Counter(grade(*gt_areas(p, "train")[2]) for p in sorted(pathlib.Path(DATA, "images/train").iterdir()))
majority = train_grades.most_common(1)[0][0]
gt_sets, pred_sets, grades, err_b, err_r = [], [], [], [], []
t0 = time.time()
for path in sorted(pathlib.Path(DATA, "images/test").iterdir()):
    res = model.predict(str(path), imgsz=DEFECT_IMGSZ, conf=CONF, device="cpu", verbose=False)[0]
    rows, potato, (gb, gr) = gt_areas(path, "test")
    cls = res.boxes.cls.cpu().numpy().astype(int)
    outlines = res.masks.xy if res.masks is not None else []
    pb, pr = (area_pct(fill_outlines(potato.shape, [o for o, c in zip(outlines, cls) if c in g]), potato) for g in (BLEMISH, ROT))
    gt_sets.append({c for c, _ in rows})
    pred_sets.append(set(cls.tolist()))
    grades.append((grade(gb, gr), grade(pb, pr)))
    err_b.append(abs(pb - gb))
    err_r.append(abs(pr - gr))
seconds = time.time() - t0
n = len(grades)
rnd = lambda x: round(x, 4) if isinstance(x, float) else x
recalls = [sum(p == g for gg, p in grades if gg == g) / k for g in "ABC" if (k := sum(gg == g for gg, _ in grades))]
accuracy = round(sum(g == p for g, p in grades) / n, 4)
grading = {
    "n_potatoes": n, "conf": CONF, "imgsz": DEFECT_IMGSZ,
    "grade_accuracy": accuracy,
    "grade_balanced_accuracy": round(sum(recalls) / len(recalls), 4),   # mean per-grade recall: always "C" scores 0.33
    "majority_grade_baseline": {"grade": majority, "accuracy": round(sum(g == majority for g, _ in grades) / n, 4),
                                "train_grade_counts": dict(train_grades)},
    "confusion_label_to_predicted": {g: dict(Counter(p for gg, p in grades if gg == g)) for g in "ABC"},
    "bruising_pct_mae": round(float(np.mean(err_b)), 2), "rot_pct_mae": round(float(np.mean(err_r)), 2),
    "presence": {DEFECT_NAMES[c]: {k: rnd(v) for k, v in d.items()} for c, d in presence(gt_sets, pred_sets, len(DEFECT_NAMES)).items()},
    "cpu_seconds_per_crop": round(seconds / n, 3),
}
print(json.dumps(grading, indent=1))


## 5. Report


In [ ]:
import platform
import PIL
report = {
    "model": "yolov11n-krishi-bd-v2.1", "stage": "defect, run 2's chosen n1024", "weights_sha256": DEFECT_SHA256,
    "why": "run 2's grading check crashed on CUDA out-of-memory; this is that pre-registered check, run once on CPU",
    "split": "test", "mask_source": "Results.masks.xy outlines rasterised at the crop's size (optical_grader.fill_outlines)",
    "target": {"grade_accuracy": TARGET}, "meets_target": accuracy >= TARGET,
    "grading": grading, "samples": sorted(p.name for p in SAMPLES.iterdir()),
    "dataset": {"name": "Diamant Potato Dataset", "doi": "10.17632/4vty9rz7zw.1", "licence": "CC BY 4.0",
                "sha256": {k: v[1] for k, v in ZIPS.items()}},
    "versions": {"python": platform.python_version(), "ultralytics": ultralytics.__version__, "torch": torch.__version__,
                 "numpy": np.__version__, "pillow": PIL.__version__, "opencv": cv2.__version__},
}
json.dump(report, open(f"{WORK}/grade_check.json", "w"), indent=2)
print(f"grade accuracy {accuracy} (target {TARGET}, majority baseline {grading['majority_grade_baseline']['accuracy']})")
